In [ ]:
"""
Domain-Specific Q&A Chatbot using TF-IDF + Cosine Similarity
Complete implementation with all requirements and bonus features
"""

import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. DATASET CREATION
# ============================================

def create_dataset():
    """Create or load the Q&A dataset"""
    
    # Initial dataset with multiple domains
    data = {
        "question": [
            # Technology
            "What is AI?",
            "What is machine learning?",
            "What is deep learning?",
            "What is Python?",
            "What is data science?",
            "How to learn AI?",
            "What is natural language processing?",
            "What is computer vision?",
            "What is neural network?",
            "What is big data?",
            
            # Health
            "What is a healthy diet?",
            "How to reduce stress?",
            "What are the benefits of exercise?",
            "How much water should I drink daily?",
            "What is meditation?",
            "How to improve sleep quality?",
            "What are superfoods?",
            "How to boost immune system?",
            "What is mental health?",
            "What are the signs of dehydration?",
            
            # Education
            "What is the importance of education?",
            "How to study effectively?",
            "What is online learning?",
            "How to improve memory?",
            "What is critical thinking?",
            "How to manage time wisely?",
            "What is STEM education?",
            "How to develop leadership skills?",
            "What are soft skills?",
            "How to prepare for exams?",
            
            # General Knowledge
            "What is the capital of France?",
            "What is the largest ocean?",
            "Who invented the telephone?",
            "What is the speed of light?",
            "What is global warming?",
            "What is renewable energy?",
            "What is the human genome?",
            "What is the internet?",
            "What is artificial intelligence?",
            "What is climate change?"
        ],
        "answer": [
            # Technology answers
            "AI (Artificial Intelligence) is the simulation of human intelligence in machines that are programmed to think and learn like humans.",
            "Machine learning is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed.",
            "Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn from data.",
            "Python is a high-level, interpreted programming language known for its simplicity and readability, widely used in AI, data science, and web development.",
            "Data science is an interdisciplinary field that uses scientific methods, algorithms, and systems to extract insights from data.",
            "Start with Python programming, then learn machine learning basics, followed by deep learning and AI concepts. Practice with projects and online courses.",
            "Natural Language Processing (NLP) is a branch of AI that helps computers understand, interpret, and manipulate human language.",
            "Computer vision is a field of AI that enables computers to interpret and understand visual information from images and videos.",
            "A neural network is a series of algorithms that mimic the human brain to recognize patterns and relationships in data.",
            "Big data refers to extremely large datasets that traditional data processing software can't handle, requiring advanced analytics techniques.",
            
            # Health answers
            "A healthy diet includes a variety of foods from all food groups: fruits, vegetables, whole grains, lean proteins, and healthy fats, while limiting processed foods and added sugars.",
            "Reduce stress through regular exercise, meditation, adequate sleep, time management, and practicing mindfulness or deep breathing exercises.",
            "Exercise benefits include improved cardiovascular health, weight management, better mood, increased energy, better sleep, and reduced risk of chronic diseases.",
            "The general recommendation is to drink 8 glasses (about 2 liters) of water daily, but needs vary based on activity level, climate, and individual factors.",
            "Meditation is a practice where an individual uses techniques like mindfulness or focusing the mind on a particular object, thought, or activity to achieve mental clarity.",
            "Improve sleep quality by maintaining a consistent schedule, creating a relaxing bedtime routine, limiting screen time before bed, and keeping the bedroom dark and cool.",
            "Superfoods are nutrient-rich foods considered especially beneficial for health and well-being, including berries, leafy greens, nuts, and fatty fish.",
            "Boost your immune system through proper nutrition, regular exercise, adequate sleep, stress management, and staying hydrated.",
            "Mental health refers to cognitive, behavioral, and emotional well-being. It's about how people think, feel, and behave in daily life.",
            "Signs of dehydration include thirst, dry mouth, dark urine, fatigue, dizziness, and decreased urine output.",
            
            # Education answers
            "Education is important as it provides knowledge, skills, and critical thinking abilities essential for personal development, career success, and societal contribution.",
            "Study effectively by using active recall, spaced repetition, creating a conducive environment, taking regular breaks, and understanding concepts rather than memorizing.",
            "Online learning is education that takes place over the internet, offering flexible, accessible, and often self-paced learning opportunities.",
            "Improve memory through regular exercise, adequate sleep, mental stimulation, healthy diet, and using mnemonic devices or visualization techniques.",
            "Critical thinking is the objective analysis and evaluation of an issue to form a judgment, involving skills like reasoning, analysis, and problem-solving.",
            "Manage time wisely by prioritizing tasks, setting goals, using calendars, avoiding procrastination, and breaking large tasks into manageable chunks.",
            "STEM education is an interdisciplinary approach to learning that focuses on Science, Technology, Engineering, and Mathematics subjects.",
            "Develop leadership skills through communication, empathy, decision-making practice, continuous learning, and seeking mentorship or leadership opportunities.",
            "Soft skills are interpersonal skills like communication, teamwork, adaptability, problem-solving, and emotional intelligence that are crucial in workplace success.",
            "Prepare for exams by starting early, creating a study schedule, practicing with past papers, studying actively, and getting adequate rest before the exam.",
            
            # General Knowledge answers
            "The capital of France is Paris.",
            "The largest ocean on Earth is the Pacific Ocean.",
            "Alexander Graham Bell invented the telephone in 1876.",
            "The speed of light is approximately 299,792,458 meters per second (about 186,282 miles per second).",
            "Global warming is the long-term heating of Earth's surface observed since the pre-industrial period due to human activities, primarily fossil fuel burning.",
            "Renewable energy comes from natural sources that are constantly replenished, including solar, wind, hydroelectric, geothermal, and biomass energy.",
            "The human genome is the complete set of human genetic information, containing approximately 3 billion base pairs of DNA, organized into 23 pairs of chromosomes.",
            "The internet is a global network of interconnected computers that uses standardized protocols to share information and provide communication services worldwide.",
            "Artificial intelligence is the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction.",
            "Climate change refers to significant, long-term changes in the global climate, including temperature, precipitation, and wind patterns, largely driven by human activities."
        ]
    }
    
    df = pd.DataFrame(data)
    return df

# ============================================
# 2. TEXT PREPROCESSING
# ============================================

def preprocess(text):
    """Clean and preprocess text for better matching"""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# ============================================
# 3. CHATBOT CLASS
# ============================================

class QAChatbot:
    def __init__(self, dataset=None, similarity_threshold=0.3):
        """Initialize the chatbot with dataset and configuration"""
        self.similarity_threshold = similarity_threshold
        self.df = dataset if dataset is not None else create_dataset()
        self.vectorizer = None
        self.X = None
        self._prepare_data()
        
    def _prepare_data(self):
        """Prepare the data with preprocessing and TF-IDF vectorization"""
        # Preprocess questions
        self.df['clean_question'] = self.df['question'].apply(preprocess)
        
        # Add n-grams (unigrams and bigrams) for better matching
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),  # Both unigrams and bigrams
            stop_words='english',
            min_df=1,
            max_features=None
        )
        
        # Fit and transform the clean questions
        self.X = self.vectorizer.fit_transform(self.df['clean_question'])
        
        print(f" Dataset loaded with {len(self.df)} Q&A pairs")
        print(f" Vector dimensions: {self.X.shape}")
        print("=" * 50)
    
    def get_response(self, query):
        """Get the most relevant answer for a given query"""
        if not query or not isinstance(query, str):
            return "Please ask a valid question."
        
        # Preprocess the query
        clean_query = preprocess(query)
        
        # Transform query to vector
        query_vec = self.vectorizer.transform([clean_query])
        
        # Calculate cosine similarity with all questions
        similarities = cosine_similarity(query_vec, self.X).flatten()
        
        # Get the best match index and similarity score
        best_index = similarities.argmax()
        best_score = similarities[best_index]
        
        # Check if similarity meets threshold
        if best_score < self.similarity_threshold:
            return f"I'm not sure about that. The closest match (confidence: {best_score:.2f}) is below my confidence threshold. Could you rephrase your question?"
        
        # Return the answer with confidence score
        answer = self.df.iloc[best_index]['answer']
        
        # Add matched question for transparency
        matched_question = self.df.iloc[best_index]['question']
        confidence = f"{best_score:.2%}"
        
        # Return with optional details
        return f"{answer} (Confidence: {confidence})"
    
    def get_answer_with_context(self, query):
        """Get answer with additional context about the match"""
        if not query or not isinstance(query, str):
            return "Please ask a valid question."
        
        clean_query = preprocess(query)
        query_vec = self.vectorizer.transform([clean_query])
        similarities = cosine_similarity(query_vec, self.X).flatten()
        
        best_index = similarities.argmax()
        best_score = similarities[best_index]
        
        if best_score < self.similarity_threshold:
            return {
                'answer': "I don't have enough confidence to answer this question.",
                'confidence': best_score,
                'matched_question': None,
                'threshold': self.similarity_threshold
            }
        
        return {
            'answer': self.df.iloc[best_index]['answer'],
            'confidence': best_score,
            'matched_question': self.df.iloc[best_index]['question'],
            'threshold': self.similarity_threshold
        }
    
    def get_top_matches(self, query, top_n=3):
        """Get top N matches for a query"""
        clean_query = preprocess(query)
        query_vec = self.vectorizer.transform([clean_query])
        similarities = cosine_similarity(query_vec, self.X).flatten()
        
        # Get top N indices
        top_indices = similarities.argsort()[-top_n:][::-1]
        
        matches = []
        for idx in top_indices:
            if similarities[idx] >= self.similarity_threshold:
                matches.append({
                    'question': self.df.iloc[idx]['question'],
                    'answer': self.df.iloc[idx]['answer'],
                    'similarity': similarities[idx]
                })
        
        return matches

# ============================================
# 4. MAIN CHATBOT LOOP
# ============================================

def run_chatbot():
    """Run the interactive chatbot in terminal"""
    print("\n" + "=" * 60)
    print(" DOMAIN-SPECIFIC Q&A CHATBOT ")
    print("=" * 60)
    print("Welcome! I can answer questions about:")
    print("• Technology (AI, ML, Programming, etc.)")
    print("• Health & Wellness")
    print("• Education & Learning")
    print("• General Knowledge")
    print("\nHow to use:")
    print("• Type your question and I'll find the best answer")
    print("• Type 'exit' or 'quit' to end the conversation")
    print("• Type 'info' to see all available topics")
    print("=" * 60 + "\n")
    
    # Initialize chatbot
    chatbot = QAChatbot(similarity_threshold=0.2)  # Lower threshold for demo
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
            
            if not user_input:
                continue
                
            if user_input.lower() in ['exit', 'quit', 'bye']:
                print("\n Bot: Goodbye! Have a great day! ")
                break
            
            if user_input.lower() == 'info':
                print("\n Bot: I can answer questions about technology, health, education, and general knowledge.")
                print("Example questions:")
                print("• What is AI?")
                print("• How to reduce stress?")
                print("• What is the importance of education?")
                print("• What is the capital of France?")
                continue
            
            # Get response
            response = chatbot.get_response(user_input)
            print(f"\n Bot: {response}")
        
            # Show alternatives if confidence is low
            if "below my confidence threshold" in response:
                matches = chatbot.get_top_matches(user_input, top_n=2)
                if matches:
                    print("\n Did you mean one of these?")
                    for i, match in enumerate(matches, 1):
                        print(f"   {i}. {match['question']} (Confidence: {match['similarity']:.2%})")
            
        except KeyboardInterrupt:
            print("\n\n Bot: Goodbye! ")
            break
        except Exception as e:
            print(f"\n Error: {str(e)}")
            print("Please try asking your question differently.")

# ============================================
# 5. SAVE DATASET TO CSV
# ============================================

def save_dataset():
    """Save the dataset to CSV file"""
    try:
        df = create_dataset()
        df.to_csv('qa_dataset.csv', index=False)
        print(" Dataset saved to 'qa_dataset.csv'")
        print(f" Total Q&A pairs: {len(df)}")
        return df
    except Exception as e:
        print(f" Error saving dataset: {e}")
        return None

# ============================================
# 6. DEMONSTRATION FUNCTION
# ============================================

def demo_chatbot():
    """Demonstrate chatbot capabilities with sample queries"""
    print("\n" + "=" * 60)
    print(" CHATBOT DEMONSTRATION ")
    print("=" * 60)
    
    chatbot = QAChatbot(similarity_threshold=0.2)
    
    test_queries = [
        "what is AI",
        "machine learning basics",
        "how to reduce stress",
        "what is Python",
        "capital of France",
        "benefits of exercise",
        "how to study effectively",
        "what is global warming",
        "random question about aliens"  # This should trigger low confidence
    ]
    
    print("\nTesting with various queries:\n")
    print("-" * 60)
    
    for query in test_queries:
        print(f"\nQ: {query}")
        response = chatbot.get_response(query)
        print(f"A: {response}")
        
        # Show context for the first few queries
        if len(query) < 15:
            context = chatbot.get_answer_with_context(query)
            if context['matched_question']:
                print(f"   ℹ Matched: '{context['matched_question']}'")
                print(f"    Confidence: {context['confidence']:.2%}")
        print("-" * 40)

# ============================================
# 7. MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print(" DOMAIN-SPECIFIC Q&A CHATBOT APPLICATION ")
    print("=" * 60)
    
    # Save dataset to CSV
    save_dataset()
    
    print("\n Choose an option:")
    print("1. Run demonstration (sample questions)")
    print("2. Run interactive chatbot")
    print("3. Exit")
    
    choice = input("\nEnter your choice (1, 2, or 3): ").strip()
    
    if choice == '1':
        demo_chatbot()
    elif choice == '2':
        run_chatbot()
    elif choice == '3':
        print("\n Exiting. Have a great day!")
    else:
        print("\n Invalid choice. Running interactive chatbot by default...")
        run_chatbot()

 DOMAIN-SPECIFIC Q&A CHATBOT APPLICATION 
 Dataset saved to 'qa_dataset.csv'
 Total Q&A pairs: 40

 Choose an option:
1. Run demonstration (sample questions)
2. Run interactive chatbot
3. Exit

 DOMAIN-SPECIFIC Q&A CHATBOT 
Welcome! I can answer questions about:
• Technology (AI, ML, Programming, etc.)
• Health & Wellness
• Education & Learning
• General Knowledge

How to use:
• Type your question and I'll find the best answer
• Type 'exit' or 'quit' to end the conversation
• Type 'info' to see all available topics

 Dataset loaded with 40 Q&A pairs
 Vector dimensions: (40, 113)

 Bot: The capital of France is Paris. (Confidence: 100.00%)

 Bot: AI (Artificial Intelligence) is the simulation of human intelligence in machines that are programmed to think and learn like humans. (Confidence: 73.58%)

 Bot: AI (Artificial Intelligence) is the simulation of human intelligence in machines that are programmed to think and learn like humans. (Confidence: 73.58%)

 Bot: Online learning is educa

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e56cc519-39a7-4783-b362-37a8abe8272f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>